# 🐄 Livestock Government Scheme Advisor — Model Training (Google Colab)

This notebook trains **two models**:

1. **Stage 1 — Breed Classifier (CNN / image)**: identifies the animal breed from a photo, using transfer learning (MobileNetV2, **PyTorch/torchvision**) on four Kaggle datasets (Indian Bovine Breeds, Indian Cattle Image Dataset, Indian Buffalo Dataset, Sheep Breed Classification).
2. **Stage 2 — Scheme Eligibility Model (tabular / multi-label)**: given breed + farmer details (animals owned, social category, land holding, employment status, gender, district), predicts which of the 16 Maharashtra + Central livestock schemes the farmer is eligible for.

**⚠️ Important data-coverage note before you start:**
The four Kaggle image datasets cover **cattle, buffalo and sheep** breeds only. They do **not** contain goat or poultry images. Our Stage-2 CSV includes goat breeds (Osmanabadi, Sangamneri, Local goat) and poultry classes (Desi poultry, Kadaknath, Improved layer/broiler). So:
- The photo-based breed classifier will only ever output a **cattle / buffalo / sheep** breed.
- The frontend has a **manual "select species/breed" fallback** for goat and poultry owners (a farmer with goats/chickens just picks their animal from a dropdown instead of uploading a photo).
- If you later find/label goat & poultry image datasets, just add them to `DATASET_CONFIGS` in Section 2 and retrain — nothing else changes.

**Why PyTorch and not TensorFlow:** TensorFlow currently has no installable build for Python 3.14 at all (even on your own machine later, if you run inference outside Colab). PyTorch does ship 3.14-compatible wheels, so Stage 1 uses `torchvision` — this keeps the backend runnable on whatever Python version you have installed, no separate environment required.

**Runtime:** Runtime → Change runtime type → **T4 GPU** (or better) before running. Colab itself uses its own Python version internally — that's independent of whatever Python you have on your own laptop for the backend later.


## 1. Setup — install & import libraries

In [ ]:
!pip install -q kaggle torch torchvision "scikit-learn==1.8.0" joblib pandas numpy matplotlib seaborn pillow


In [ ]:
import os, json, shutil, zipfile, random, glob, pathlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (accuracy_score, f1_score, hamming_loss,
                              classification_report)
import joblib

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch version:", torch.__version__)
print("Device:", DEVICE)


## 2. Kaggle API setup

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token**. This downloads a `kaggle.json` file.
2. Run the cell below and upload that `kaggle.json` when prompted.


In [ ]:
from google.colab import files
print("Upload your kaggle.json (from kaggle.com/settings -> API -> Create New Token)")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials configured.")


## 3. Download the four breed-image datasets

In [ ]:
RAW_DIR = '/content/raw_datasets'
os.makedirs(RAW_DIR, exist_ok=True)

DATASET_CONFIGS = [
    # (kaggle slug, local folder name, species this dataset covers)
    ("lukex9442/indian-bovine-breeds",        "indian_bovine_breeds",       "cattle"),
    ("atharvadarpude/indian-cattle-image-dataset", "indian_cattle_alt",     "cattle"),
    ("atharvadarpude/indian-buffalo-dataset",  "indian_buffalo",             "buffalo"),
    ("divyansh22/sheep-breed-classification",  "sheep_breeds",               "sheep"),
]

for slug, folder, species in DATASET_CONFIGS:
    dest = os.path.join(RAW_DIR, folder)
    if os.path.isdir(dest) and len(os.listdir(dest)) > 0:
        print(f"[skip] {slug} already downloaded")
        continue
    print(f"Downloading {slug} ...")
    !kaggle datasets download -d {slug} -p {RAW_DIR} --unzip -q
    # kaggle unzips into RAW_DIR directly (mixed with others), so we move the
    # newest top-level entries into their own folder. Simplest robust approach:
    # download into a dedicated subfolder instead.


In [ ]:
# Re-do downloads cleanly, each into its own subfolder (safer than mixing them)
import subprocess

def kaggle_download(slug, dest):
    os.makedirs(dest, exist_ok=True)
    if len(os.listdir(dest)) > 0:
        print(f"[skip] {slug} -> {dest} (already has files)")
        return
    print(f"Downloading {slug} -> {dest}")
    subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", dest, "--unzip", "-q"], check=True)

for slug, folder, species in DATASET_CONFIGS:
    kaggle_download(slug, os.path.join(RAW_DIR, folder))

for slug, folder, species in DATASET_CONFIGS:
    d = os.path.join(RAW_DIR, folder)
    print(folder, '->', os.listdir(d)[:10])


## 4. Auto-discover class folders & map to canonical breed names

Kaggle datasets rarely have identical folder-naming conventions. This cell walks each
dataset, finds the deepest folders that actually contain images, and lets you see what
breed names exist so they can be mapped onto the **canonical breed labels used in the
Stage-2 CSV**: `Dangi, Deccani, Deoni, Gaolao, Gir, Red Kandhari, Sahiwal, Crossbred (HF/Jersey),
Buffalo (non-descript), Pandharpuri, Local sheep, Madgyal`.


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_image_class_folders(root):
    """Return {folder_path: image_count} for every folder that directly contains images."""
    result = {}
    for dirpath, dirnames, filenames in os.walk(root):
        n_images = sum(1 for f in filenames if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
        if n_images > 0:
            result[dirpath] = n_images
    return result

for slug, folder, species in DATASET_CONFIGS:
    root = os.path.join(RAW_DIR, folder)
    class_folders = find_image_class_folders(root)
    print(f"\n=== {folder} ({species}) — {len(class_folders)} folders with images ===")
    for p, n in sorted(class_folders.items()):
        print(f"  {n:5d}  {p}")


In [ ]:
# CANONICAL_BREEDS = breed labels that appear in the Stage-2 eligibility CSV,
# restricted to species our image datasets can actually cover (cattle/buffalo/sheep).
CANONICAL_BREEDS = [
    "Dangi", "Deccani", "Deoni", "Gaolao", "Gir", "Red Kandhari", "Sahiwal",
    "Crossbred (HF/Jersey)", "Buffalo (non-descript)", "Pandharpuri",
    "Local sheep", "Madgyal",
]

# Aliases: raw folder-name substrings (lowercased) -> canonical breed.
# EDIT THIS after inspecting the printed folder names above — kaggle dataset
# class names vary by dataset (e.g. "Red_Kandhari", "red-kandhari-cattle").
BREED_ALIASES = {
    "dangi": "Dangi",
    "deccani": "Deccani",
    "deoni": "Deoni",
    "gaolao": "Gaolao",
    "gir": "Gir",
    "kandhari": "Red Kandhari",
    "sahiwal": "Sahiwal",
    "crossbred": "Crossbred (HF/Jersey)",
    "jersey": "Crossbred (HF/Jersey)",
    "hf": "Crossbred (HF/Jersey)",
    "murrah": "Buffalo (non-descript)",   # fallback bucket for buffalo breeds not in our label set
    "buffalo": "Buffalo (non-descript)",
    "pandharpuri": "Pandharpuri",
    "madgyal": "Madgyal",
    # sheep dataset classes that don't map to a canonical breed fall back to "Local sheep"
}

def map_folder_to_breed(folder_path, species):
    name = os.path.basename(folder_path).lower().replace('-', ' ').replace('_', ' ')
    for key, canon in BREED_ALIASES.items():
        if key in name:
            return canon
    # species-level fallback so every image dataset still contributes examples
    if species == "sheep":
        return "Local sheep"
    if species == "buffalo":
        return "Buffalo (non-descript)"
    return None  # unmapped cattle sub-breed -> excluded (keeps label set clean)


## 5. Build one unified image folder (`/content/breed_dataset/<canonical_breed>/*.jpg`)

In [ ]:
UNIFIED_DIR = '/content/breed_dataset'
if os.path.isdir(UNIFIED_DIR):
    shutil.rmtree(UNIFIED_DIR)
os.makedirs(UNIFIED_DIR, exist_ok=True)

counter = {b: 0 for b in CANONICAL_BREEDS}
skipped_folders = []

for slug, folder, species in DATASET_CONFIGS:
    root = os.path.join(RAW_DIR, folder)
    class_folders = find_image_class_folders(root)
    for src_folder, n_images in class_folders.items():
        breed = map_folder_to_breed(src_folder, species)
        if breed is None:
            skipped_folders.append(src_folder)
            continue
        out_dir = os.path.join(UNIFIED_DIR, breed)
        os.makedirs(out_dir, exist_ok=True)
        for fname in os.listdir(src_folder):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in IMAGE_EXTS:
                continue
            src = os.path.join(src_folder, fname)
            dst = os.path.join(out_dir, f"{folder}_{counter[breed]:06d}{ext}")
            try:
                shutil.copy(src, dst)
                counter[breed] += 1
            except Exception as e:
                pass

print("Images per canonical breed:")
for b, n in counter.items():
    print(f"  {b:28s} {n}")
print(f"\n{len(skipped_folders)} source folders were skipped (no alias match). "
      f"Add entries to BREED_ALIASES above and re-run Sections 4-5 if you want to keep them.")

# Drop breeds with too few images to train on reliably
MIN_IMAGES_PER_CLASS = 40
final_breeds = [b for b, n in counter.items() if n >= MIN_IMAGES_PER_CLASS]
for b in list(counter):
    if b not in final_breeds:
        print(f"Dropping class '{b}' — only {counter[b]} images (< {MIN_IMAGES_PER_CLASS})")
        shutil.rmtree(os.path.join(UNIFIED_DIR, b), ignore_errors=True)

print("\nFinal breed classes used for training:", final_breeds)


## 6. Build PyTorch datasets & dataloaders (train/val split, augmentation)

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Validation/inference uses NO augmentation, only resize + normalize.
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ImageFolder applies ONE transform to the whole dataset, but we want different
# transforms for train vs val, so we load it twice (cheap - just indexes files)
# and split with matching indices.
full_dataset = datasets.ImageFolder(UNIFIED_DIR)
BREED_CLASS_NAMES = full_dataset.classes
print("Classes:", BREED_CLASS_NAMES)

n_val = int(0.2 * len(full_dataset))
n_train = len(full_dataset) - n_val
train_indices, val_indices = random_split(
    range(len(full_dataset)), [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED))

train_dataset_full = datasets.ImageFolder(UNIFIED_DIR, transform=train_transform)
val_dataset_full = datasets.ImageFolder(UNIFIED_DIR, transform=eval_transform)

train_dataset = torch.utils.data.Subset(train_dataset_full, list(train_indices))
val_dataset = torch.utils.data.Subset(val_dataset_full, list(val_indices))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train images: {len(train_dataset)}  |  Val images: {len(val_dataset)}")


## 7. Build the model — MobileNetV2 transfer learning (torchvision)

In [ ]:
def build_breed_model(num_classes):
    weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
    model = models.mobilenet_v2(weights=weights)
    for param in model.features.parameters():
        param.requires_grad = False  # freeze the convolutional backbone for phase 1

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model.to(DEVICE)

breed_model = build_breed_model(len(BREED_CLASS_NAMES))
print(breed_model.classifier)


## 8. Train — Phase 1 (frozen backbone) then Phase 2 (fine-tune top layers)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total


def fit(model, train_loader, val_loader, epochs, lr, patience=3):
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss, best_state, stale = float("inf"), None, 0

    for epoch in range(epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        history["loss"].append(train_loss); history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss); history["val_accuracy"].append(val_acc)
        print(f"Epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f} train_acc={train_acc:.4f}"
              f"  val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss, best_state, stale = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= patience:
                print(f"Early stopping (no val_loss improvement for {patience} epochs)")
                break

    if best_state is not None:
        model.load_state_dict(best_state)  # restore best weights, like Keras' restore_best_weights
    return history

EPOCHS_PHASE1 = 8
history1 = fit(breed_model, train_loader, val_loader, epochs=EPOCHS_PHASE1, lr=1e-3)


In [ ]:
# Phase 2: unfreeze the last few backbone blocks and fine-tune with a low LR
UNFREEZE_LAST_N = 40  # number of backbone parameter tensors to unfreeze, counting from the end
backbone_params = list(breed_model.features.parameters())
for param in backbone_params[-UNFREEZE_LAST_N:]:
    param.requires_grad = True

EPOCHS_PHASE2 = 6
history2 = fit(breed_model, train_loader, val_loader, epochs=EPOCHS_PHASE2, lr=1e-5)


In [ ]:
def plot_history(h1, h2, key):
    vals = h1[key] + h2[key]
    val_vals = h1[f'val_{key}'] + h2[f'val_{key}']
    plt.plot(vals, label=f'train_{key}')
    plt.plot(val_vals, label=f'val_{key}')
    plt.axvline(len(h1[key]) - 1, color='gray', linestyle='--', label='fine-tune start')
    plt.legend(); plt.title(key); plt.xlabel('epoch'); plt.show()

plot_history(history1, history2, 'accuracy')
plot_history(history1, history2, 'loss')


## 9. Evaluate & save the breed classifier

Saved as a plain **state_dict** (`breed_classifier_weights.pt`) — just the trained numbers,
no bundled architecture. This deliberately avoids `torch.jit`/TorchScript, which PyTorch
itself currently warns is unreliable on Python 3.14+ ("switch to `torch.export`"). A plain
`state_dict` only needs ordinary `torch.save`/`torch.load`, which has no such caveat — the
backend just rebuilds the same MobileNetV2 architecture in code and loads these weights into
it (see `backend/app.py`).


In [ ]:
_, val_acc = run_epoch(breed_model, val_loader, nn.CrossEntropyLoss())
print(f"Final validation accuracy: {val_acc:.3f}")

MODELS_DIR = '/content/models'
os.makedirs(MODELS_DIR, exist_ok=True)

breed_model.eval()
torch.save(breed_model.state_dict(), os.path.join(MODELS_DIR, 'breed_classifier_weights.pt'))

with open(os.path.join(MODELS_DIR, 'breed_labels.json'), 'w') as f:
    json.dump({
        "classes": BREED_CLASS_NAMES,
        "img_size": [IMG_SIZE, IMG_SIZE],
        "mean": IMAGENET_MEAN,
        "std": IMAGENET_STD,
        "architecture": "mobilenet_v2",
    }, f, indent=2)

print("Saved breed_classifier_weights.pt and breed_labels.json")


---
## 10. Stage 2 — Scheme Eligibility Model (tabular, multi-label)

Load the CSV you uploaded (`stage2_scheme_eligibility_slim.csv`). It already has a
`split` column (train/val/test) and one binary `elig_<SCHEME_CODE>` column per scheme —
we train a **multi-label classifier** that predicts the full eligibility vector (which of
the 16 schemes apply) from the farmer's profile.


In [ ]:
from google.colab import files
print("Upload stage2_scheme_eligibility_slim.csv (or stage2_scheme_eligibility.csv)")
up = files.upload()
csv_path = list(up.keys())[0]

df = pd.read_csv(csv_path)
print(df.shape)
df.head()


In [ ]:
FEATURE_COLS = ['breed', 'animals_owned', 'social_category', 'land_holding',
                'employment_status', 'gender', 'district']
CATEGORICAL_COLS = ['breed', 'social_category', 'land_holding', 'employment_status', 'gender', 'district']
NUMERIC_COLS = ['animals_owned']

LABEL_COLS = [c for c in df.columns if c.startswith('elig_')]
print(f"{len(LABEL_COLS)} scheme labels:", LABEL_COLS)

# Derive breed -> species map (used by the backend to short-circuit goat/poultry logic)
BREED_SPECIES_MAP = df.drop_duplicates('breed').set_index('breed')['species'].to_dict()
print(BREED_SPECIES_MAP)


In [ ]:
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'val']
test_df  = df[df['split'] == 'test']
print(len(train_df), len(val_df), len(test_df))

X_train, y_train = train_df[FEATURE_COLS], train_df[LABEL_COLS].astype(int)
X_val,   y_val   = val_df[FEATURE_COLS],   val_df[LABEL_COLS].astype(int)
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[LABEL_COLS].astype(int)


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_COLS),
    ('num', 'passthrough', NUMERIC_COLS),
])

base_clf = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=2,
    random_state=SEED, n_jobs=-1, class_weight='balanced_subsample')

eligibility_pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('clf', MultiOutputClassifier(base_clf, n_jobs=-1)),
])

eligibility_pipeline.fit(X_train, y_train)
print("Trained.")


In [ ]:
def evaluate_split(name, X, y):
    pred = eligibility_pipeline.predict(X)
    print(f"--- {name} ---")
    print(f"Subset accuracy (exact eligibility vector match): {accuracy_score(y, pred):.4f}")
    print(f"Hamming loss (per-label error rate):               {hamming_loss(y, pred):.4f}")
    print(f"Macro F1 (per-label):                               {f1_score(y, pred, average='macro', zero_division=0):.4f}")
    per_label_acc = (pred == y.values).mean(axis=0)
    report = pd.DataFrame({'label': LABEL_COLS, 'accuracy': per_label_acc}).sort_values('accuracy')
    print(report.to_string(index=False))
    return pred

_ = evaluate_split('VALIDATION', X_val, y_val)
test_pred = evaluate_split('TEST', X_test, y_test)


### 10b. Convenience model: predict the single **top recommended scheme** (`top1`)

This is a plain multiclass classifier on the `top1` column, handy for a headline
recommendation in the UI (in addition to the full eligibility list above).


In [ ]:
from sklearn.preprocessing import LabelEncoder

top1_encoder = LabelEncoder()
y_top1_train = top1_encoder.fit_transform(train_df['top1'])
y_top1_val   = top1_encoder.transform(val_df['top1'])
y_top1_test  = top1_encoder.transform(test_df['top1'])

top1_pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1,
                                    class_weight='balanced_subsample')),
])
top1_pipeline.fit(X_train, y_top1_train)

val_acc_top1 = accuracy_score(y_top1_val, top1_pipeline.predict(X_val))
test_acc_top1 = accuracy_score(y_top1_test, top1_pipeline.predict(X_test))
print(f"top1 val accuracy: {val_acc_top1:.4f}  |  test accuracy: {test_acc_top1:.4f}")


## 11. Save Stage-2 artifacts

In [ ]:
joblib.dump(eligibility_pipeline, os.path.join(MODELS_DIR, 'eligibility_model.joblib'))
joblib.dump(top1_pipeline, os.path.join(MODELS_DIR, 'top1_model.joblib'))
joblib.dump(top1_encoder, os.path.join(MODELS_DIR, 'top1_label_encoder.joblib'))

with open(os.path.join(MODELS_DIR, 'feature_metadata.json'), 'w') as f:
    json.dump({
        "feature_cols": FEATURE_COLS,
        "categorical_cols": CATEGORICAL_COLS,
        "numeric_cols": NUMERIC_COLS,
        "label_cols": LABEL_COLS,
        "breed_species_map": BREED_SPECIES_MAP,
        "breed_options": sorted(df['breed'].unique().tolist()),
        "social_category_options": sorted(df['social_category'].unique().tolist()),
        "land_holding_options": sorted(df['land_holding'].unique().tolist()),
        "employment_status_options": sorted(df['employment_status'].unique().tolist()),
        "gender_options": sorted(df['gender'].unique().tolist()),
        "district_options": sorted(df['district'].unique().tolist()),
    }, f, indent=2)

# Also copy the rules manifest into models/ so the backend can enrich predictions
# with human-readable scheme names / descriptions / basis text.
print("Now upload rules_manifest.json so it ships alongside the models:")
up2 = files.upload()
manifest_name = list(up2.keys())[0]
shutil.copy(manifest_name, os.path.join(MODELS_DIR, 'rules_manifest.json'))

print("Saved all Stage-2 artifacts:")
print(os.listdir(MODELS_DIR))


## 12. Zip everything and download to your machine

In [ ]:
zip_path = '/content/livestock_models.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(MODELS_DIR):
        for fname in filenames:
            fp = os.path.join(root, fname)
            zf.write(fp, os.path.relpath(fp, MODELS_DIR))

print("Zipped:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
files.download(zip_path)


## 13. Next steps

1. Unzip `livestock_models.zip` into the `backend/models/` folder of the project (see the
   file structure in the README / chat message).
2. Run the FastAPI backend: `python -m uvicorn app:app --reload --port 8000` (from inside
   `backend/` — use `python -m uvicorn ...` rather than bare `uvicorn ...` if your OS can't
   find the `uvicorn` command directly).
3. Open `frontend/index.html` in a browser (or serve it with any static server) — it calls
   the backend at `http://localhost:8000`.
4. (Optional, later) If you find goat/poultry breed-image datasets, add them to
   `DATASET_CONFIGS` in Section 3, extend `BREED_ALIASES`, and re-run this notebook end to
   end — the backend/frontend need no code changes since they read `breed_labels.json`
   dynamically.
